# AI Ethics Fairness Benchmarks
Complete, verified pipeline: data loading, baseline models (Table 3), mitigation methods (Tables 4-6), comparison summary (Table 7), and SHAP/LIME explainability analysis (Sections 5.6-5.7).

## 1. Install Dependencies

In [1]:
!pip install -q --upgrade pip
!pip install -q aif360 fairlearn shap lime xgboost scikit-learn pandas numpy matplotlib torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 38.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 2. Load and Clean Datasets (Adult Income, COMPAS, German Credit)

In [2]:
import pandas as pd
import numpy as np

# --- Adult Income ---
columns = ["age","workclass","fnlwgt","education","education-num","marital-status",
           "occupation","relationship","race","sex","capital-gain","capital-loss",
           "hours-per-week","native-country","income"]
df_adult = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
                        names=columns, skipinitialspace=True, na_values="?")
df_adult_test = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test",
                             names=columns, skipinitialspace=True, na_values="?", skiprows=1)
df_adult_test["income"] = df_adult_test["income"].str.rstrip(".")
df_adult_full = pd.concat([df_adult, df_adult_test], ignore_index=True).dropna()
df_adult_full["income"] = df_adult_full["income"].str.strip()

# --- COMPAS ---
df_compas_raw = pd.read_csv("https://raw.githubusercontent.com/propublica/compas-analysis/master/compas-scores-two-years.csv")
df_compas = df_compas_raw[
    (df_compas_raw["days_b_screening_arrest"] <= 30) &
    (df_compas_raw["days_b_screening_arrest"] >= -30) &
    (df_compas_raw["is_recid"] != -1) &
    (df_compas_raw["c_charge_degree"] != "O") &
    (df_compas_raw["score_text"] != "N/A")
].copy()
df_compas = df_compas[df_compas["race"].isin(["African-American", "Caucasian"])].copy()
compas_cols = ["age","age_cat","sex","race","juv_fel_count","juv_misd_count",
               "juv_other_count","priors_count","c_charge_degree","two_year_recid"]
df_compas_clean = df_compas[compas_cols].copy().dropna()

# --- German Credit ---
columns_german = ["checking_account_status","duration_months","credit_history","purpose",
                   "credit_amount","savings_account","employment_since","installment_rate",
                   "personal_status_sex","other_debtors","residence_since","property","age",
                   "other_installment_plans","housing","existing_credits","job",
                   "num_dependents","telephone","foreign_worker","credit_risk"]
df_german = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/statlog/german/german.data",
                          names=columns_german, sep=r"\s+")
df_german["age_group"] = np.where(df_german["age"] >= 25, "old", "young")

print("Adult:", df_adult_full.shape)
print("COMPAS:", df_compas_clean.shape)
print("German:", df_german.shape)

Adult: (45222, 15)
COMPAS: (5278, 10)
German: (1000, 22)


## 3. Preprocessing and Fairness Metric Functions

In [3]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score, balanced_accuracy_score

def preprocess(df, target_col, protected_col, positive_label, privileged_value):
    df = df.copy()
    y = (df[target_col] == positive_label).astype(int)
    protected = (df[protected_col] == privileged_value).astype(int)
    X = df.drop(columns=[target_col])
    for col in X.select_dtypes(include="object").columns:
        X[col] = LabelEncoder().fit_transform(X[col])
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=X.columns)
    return X_scaled, y, protected

def fairness_metrics(y_true, y_pred, protected):
    priv_mask = protected == 1
    unpriv_mask = protected == 0

    def rate(mask, preds):
        return preds[mask].mean() if mask.sum() > 0 else np.nan

    sel_priv = rate(priv_mask, y_pred)
    sel_unpriv = rate(unpriv_mask, y_pred)
    DI = sel_unpriv / sel_priv if sel_priv > 0 else np.nan
    DPD = sel_unpriv - sel_priv

    def tpr_fpr(mask):
        yt, yp = y_true[mask], y_pred[mask]
        tp = ((yt == 1) & (yp == 1)).sum()
        fn = ((yt == 1) & (yp == 0)).sum()
        fp = ((yt == 0) & (yp == 1)).sum()
        tn = ((yt == 0) & (yp == 0)).sum()
        tpr = tp / (tp + fn) if (tp + fn) > 0 else np.nan
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan
        return tpr, fpr

    tpr_priv, fpr_priv = tpr_fpr(priv_mask)
    tpr_unpriv, fpr_unpriv = tpr_fpr(unpriv_mask)

    EOD = max(abs(tpr_unpriv - tpr_priv), abs(fpr_unpriv - fpr_priv))
    AOD = 0.5 * ((tpr_unpriv - tpr_priv) + (fpr_unpriv - fpr_priv))
    return DI, DPD, EOD, AOD

X_adult, y_adult, prot_adult = preprocess(df_adult_full, "income", "sex", ">50K", "Male")
X_compas, y_compas, prot_compas = preprocess(df_compas_clean, "two_year_recid", "race", 1, "Caucasian")
X_german, y_german, prot_german = preprocess(df_german, "credit_risk", "age_group", 1, "old")

print("Preprocessing complete.")

Preprocessing complete.


## 4. Table 3 — Baseline Models with 5-Fold Cross-Validation

In [4]:
def run_experiment_cv(X, y, protected, dataset_name, n_splits=5):
    models = {
        "LR": LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs"),
        "RF": RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42),
        "XGB": XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6, subsample=0.8,
                              eval_metric="logloss", random_state=42)
    }
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    X_arr = X.reset_index(drop=True)
    y_arr = y.reset_index(drop=True)
    prot_arr = protected.reset_index(drop=True)
    results = []
    for model_name, model in models.items():
        fold_metrics = {"Accuracy": [], "AUC-ROC": [], "F1": [], "Bal.Acc": [],
                         "DI": [], "DPD": [], "EOD": [], "AOD": []}
        for train_idx, test_idx in skf.split(X_arr, y_arr):
            X_train, X_test = X_arr.iloc[train_idx], X_arr.iloc[test_idx]
            y_train, y_test = y_arr.iloc[train_idx], y_arr.iloc[test_idx]
            prot_test = prot_arr.iloc[test_idx]
            model.fit(X_train, y_train)
            y_pred = model.predict(X_test)
            y_proba = model.predict_proba(X_test)[:, 1]
            fold_metrics["Accuracy"].append(accuracy_score(y_test, y_pred))
            fold_metrics["AUC-ROC"].append(roc_auc_score(y_test, y_proba))
            fold_metrics["F1"].append(f1_score(y_test, y_pred))
            fold_metrics["Bal.Acc"].append(balanced_accuracy_score(y_test, y_pred))
            DI, DPD, EOD, AOD = fairness_metrics(y_test.reset_index(drop=True), pd.Series(y_pred), prot_test.reset_index(drop=True))
            fold_metrics["DI"].append(DI)
            fold_metrics["DPD"].append(abs(DPD))
            fold_metrics["EOD"].append(EOD)
            fold_metrics["AOD"].append(abs(AOD))
        row = {"Dataset": dataset_name, "Model": model_name}
        for metric, values in fold_metrics.items():
            values = np.array(values)
            row[f"{metric}_mean"] = round(np.nanmean(values), 3)
            row[f"{metric}_std"] = round(np.nanstd(values), 3)
        results.append(row)
    return pd.DataFrame(results)

results_adult_cv = run_experiment_cv(X_adult, y_adult, prot_adult, "Adult")
results_compas_cv = run_experiment_cv(X_compas, y_compas, prot_compas, "COMPAS")
results_german_cv = run_experiment_cv(X_german, y_german, prot_german, "German")

final_table3_cv = pd.concat([results_adult_cv, results_compas_cv, results_german_cv], ignore_index=True)
final_table3_cv

,Dataset,Model,Accuracy_mean,Accuracy_std,AUC-ROC_mean,AUC-ROC_std,F1_mean,F1_std,Bal.Acc_mean,Bal.Acc_std,DI_mean,DI_std,DPD_mean,DPD_std,EOD_mean,EOD_std,AOD_mean,AOD_std
0,Adult,LR,0.820,0.006,0.851,0.009,0.557,0.015,0.698,0.008,0.171,0.016,0.179,0.005,0.286,0.011,0.180,0.005
1,Adult,RF,0.855,0.005,0.914,0.004,0.653,0.015,0.753,0.009,0.299,0.016,0.154,0.004,0.086,0.028,0.067,0.014
2,Adult,XGB,0.870,0.005,0.927,0.004,0.714,0.011,0.797,0.008,0.316,0.017,0.180,0.006,0.081,0.011,0.071,0.004
3,COMPAS,LR,0.675,0.011,0.729,0.014,0.636,0.013,0.671,0.011,2.152,0.262,0.283,0.031,0.318,0.031,0.249,0.027
4,COMPAS,RF,0.677,0.012,0.727,0.014,0.635,0.012,0.672,0.012,2.022,0.224,0.259,0.037,0.277,0.028,0.223,0.032
5,COMPAS,XGB,0.672,0.015,0.719,0.013,0.629,0.016,0.668,0.015,1.886,0.222,0.234,0.035,0.257,0.031,0.199,0.029
6,German,LR,0.757,0.021,0.783,0.019,0.836,0.015,0.673,0.022,0.737,0.044,0.214,0.043,0.234,0.081,0.178,0.053
7,German,RF,0.754,0.012,0.794,0.034,0.840,0.010,0.640,0.013,0.853,0.047,0.127,0.044,0.175,0.085,0.100,0.065
8,German,XGB,0.770,0.019,0.796,0.018,0.845,0.014,0.687,0.019,0.818,0.093,0.146,0.076,0.182,0.067,0.120,0.055


## 5. Table 4 — Reweighing (Pre-Processing Mitigation)

In [5]:
def compute_reweighing_weights(y, protected):
    y = y.reset_index(drop=True)
    protected = protected.reset_index(drop=True)
    n = len(y)
    weights = np.zeros(n)
    for p_val in [0, 1]:
        for y_val in [0, 1]:
            mask = (protected == p_val) & (y == y_val)
            n_group = mask.sum()
            if n_group == 0:
                continue
            p_p = (protected == p_val).mean()
            p_y = (y == y_val).mean()
            expected = p_p * p_y * n
            weights[mask] = expected / n_group
    return weights

def run_experiment_reweighted_cv(X, y, protected, dataset_name, n_splits=5):
    models = {
        "LR": LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs"),
        "RF": RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42),
        "XGB": XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6, subsample=0.8,
                              eval_metric="logloss", random_state=42)
    }
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    X_arr = X.reset_index(drop=True)
    y_arr = y.reset_index(drop=True)
    prot_arr = protected.reset_index(drop=True)
    results = []
    for model_name, model in models.items():
        fold_metrics = {"Accuracy": [], "AUC-ROC": [], "DI": [], "DPD": [], "EOD": [], "AOD": []}
        for train_idx, test_idx in skf.split(X_arr, y_arr):
            X_train, X_test = X_arr.iloc[train_idx], X_arr.iloc[test_idx]
            y_train, y_test = y_arr.iloc[train_idx], y_arr.iloc[test_idx]
            prot_train, prot_test = prot_arr.iloc[train_idx], prot_arr.iloc[test_idx]
            sample_weights = compute_reweighing_weights(y_train, prot_train)
            model.fit(X_train, y_train, sample_weight=sample_weights)
            y_pred = model.predict(X_test)
            y_proba = model.predict_proba(X_test)[:, 1]
            fold_metrics["Accuracy"].append(accuracy_score(y_test, y_pred))
            fold_metrics["AUC-ROC"].append(roc_auc_score(y_test, y_proba))
            DI, DPD, EOD, AOD = fairness_metrics(y_test.reset_index(drop=True), pd.Series(y_pred), prot_test.reset_index(drop=True))
            fold_metrics["DI"].append(DI)
            fold_metrics["DPD"].append(abs(DPD))
            fold_metrics["EOD"].append(EOD)
            fold_metrics["AOD"].append(abs(AOD))
        row = {"Dataset": dataset_name, "Model": model_name}
        for metric, values in fold_metrics.items():
            values = np.array(values)
            row[f"{metric}_mean"] = round(np.nanmean(values), 3)
            row[f"{metric}_std"] = round(np.nanstd(values), 3)
        results.append(row)
    return pd.DataFrame(results)

results_adult_rw = run_experiment_reweighted_cv(X_adult, y_adult, prot_adult, "Adult")
results_compas_rw = run_experiment_reweighted_cv(X_compas, y_compas, prot_compas, "COMPAS")
results_german_rw = run_experiment_reweighted_cv(X_german, y_german, prot_german, "German")

final_table4_reweighted = pd.concat([results_adult_rw, results_compas_rw, results_german_rw], ignore_index=True)
final_table4_reweighted

,Dataset,Model,Accuracy_mean,Accuracy_std,AUC-ROC_mean,AUC-ROC_std,DI_mean,DI_std,DPD_mean,DPD_std,EOD_mean,EOD_std,AOD_mean,AOD_std
0,Adult,LR,0.809,0.006,0.830,0.009,0.589,0.026,0.067,0.005,0.015,0.007,0.007,0.004
1,Adult,RF,0.852,0.004,0.908,0.005,0.587,0.039,0.082,0.009,0.155,0.021,0.075,0.008
2,Adult,XGB,0.864,0.003,0.922,0.004,0.563,0.036,0.098,0.010,0.141,0.012,0.067,0.008
3,COMPAS,LR,0.662,0.009,0.718,0.016,1.064,0.103,0.040,0.019,0.074,0.032,0.030,0.022
4,COMPAS,RF,0.673,0.016,0.722,0.016,1.399,0.140,0.130,0.037,0.150,0.031,0.093,0.030
5,COMPAS,XGB,0.661,0.009,0.711,0.014,1.165,0.123,0.058,0.039,0.070,0.029,0.030,0.027
6,German,LR,0.761,0.023,0.778,0.021,0.953,0.054,0.046,0.032,0.110,0.060,0.039,0.018
7,German,RF,0.755,0.017,0.789,0.030,1.030,0.060,0.046,0.031,0.116,0.093,0.068,0.052
8,German,XGB,0.772,0.021,0.793,0.021,0.850,0.065,0.120,0.054,0.138,0.040,0.078,0.046


## 6. Table 5 — Adversarial Debiasing (In-Processing Mitigation)

In [6]:
import torch
import torch.nn as nn
import torch.optim as optim

class Classifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 200), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(200, 100), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(100, 1)
        )
    def forward(self, x):
        return self.net(x)

class Adversary(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1, 50), nn.ReLU(), nn.Linear(50, 1))
    def forward(self, x):
        return self.net(x)

def adversarial_debiasing_v2(X_train, y_train, protected_train, X_test, y_test, protected_test,
                                adv_weight=0.5, epochs=100, adv_steps_per_epoch=3):
    X_train_t = torch.FloatTensor(X_train.values)
    y_train_t = torch.FloatTensor(y_train.values).unsqueeze(1)
    prot_train_t = torch.FloatTensor(protected_train.values).unsqueeze(1)
    X_test_t = torch.FloatTensor(X_test.values)

    clf = Classifier(X_train.shape[1])
    adv = Adversary()
    clf_optimizer = optim.Adam(clf.parameters(), lr=0.001)
    adv_optimizer = optim.Adam(adv.parameters(), lr=0.001)
    bce_loss = nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        for _ in range(adv_steps_per_epoch):
            clf_logits = clf(X_train_t).detach()
            adv_optimizer.zero_grad()
            adv_pred = adv(clf_logits)
            adv_loss = bce_loss(adv_pred, prot_train_t)
            adv_loss.backward()
            adv_optimizer.step()

        clf_optimizer.zero_grad()
        clf_logits = clf(X_train_t)
        clf_loss = bce_loss(clf_logits, y_train_t)
        adv_pred_for_clf = adv(clf_logits)
        adv_loss_for_clf = bce_loss(adv_pred_for_clf, prot_train_t)
        total_loss = clf_loss - adv_weight * adv_loss_for_clf
        total_loss.backward()
        clf_optimizer.step()

    with torch.no_grad():
        test_logits = clf(X_test_t)
        test_proba = torch.sigmoid(test_logits).numpy().flatten()
        test_pred = (test_proba >= 0.5).astype(int)
    return test_pred, test_proba

# Adult Income
X_train, X_test, y_train, y_test, prot_train, prot_test = train_test_split(
    X_adult, y_adult, prot_adult, test_size=0.2, stratify=y_adult, random_state=42)

print("--- Adult Income ---")
for weight in [0.7, 1.5]:
    y_pred, y_proba = adversarial_debiasing_v2(X_train, y_train, prot_train, X_test, y_test, prot_test,
                                                  adv_weight=weight, epochs=100, adv_steps_per_epoch=3)
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    DI, DPD, EOD, AOD = fairness_metrics(y_test.reset_index(drop=True), pd.Series(y_pred), prot_test.reset_index(drop=True))
    print(f"adv_weight={weight}: Acc={acc:.3f} AUC={auc:.3f} DI={DI:.3f} DPD={abs(DPD):.3f} EOD={EOD:.3f} AOD={abs(AOD):.3f}")

# COMPAS
X_train_c, X_test_c, y_train_c, y_test_c, prot_train_c, prot_test_c = train_test_split(
    X_compas, y_compas, prot_compas, test_size=0.2, stratify=y_compas, random_state=42)

print("--- COMPAS ---")
for weight in [0.5, 1.0]:
    y_pred, y_proba = adversarial_debiasing_v2(X_train_c, y_train_c, prot_train_c, X_test_c, y_test_c, prot_test_c,
                                                  adv_weight=weight, epochs=100, adv_steps_per_epoch=3)
    acc = accuracy_score(y_test_c, y_pred)
    auc = roc_auc_score(y_test_c, y_proba)
    DI, DPD, EOD, AOD = fairness_metrics(y_test_c.reset_index(drop=True), pd.Series(y_pred), prot_test_c.reset_index(drop=True))
    print(f"adv_weight={weight}: Acc={acc:.3f} AUC={auc:.3f} DI={DI:.3f} DPD={abs(DPD):.3f} EOD={EOD:.3f} AOD={abs(AOD):.3f}")

--- Adult Income ---
adv_weight=0.7: Acc=0.825 AUC=0.871 DI=0.495 DPD=0.103 EOD=0.026 AOD=0.013
adv_weight=1.5: Acc=0.824 AUC=0.860 DI=0.474 DPD=0.105 EOD=0.063 AOD=0.041
--- COMPAS ---
adv_weight=0.5: Acc=0.664 AUC=0.712 DI=1.955 DPD=0.249 EOD=0.252 AOD=0.217
adv_weight=1.0: Acc=0.654 AUC=0.687 DI=1.582 DPD=0.130 EOD=0.170 AOD=0.103


## 7. Table 6 — Equalized Odds Post-Processing

In [7]:
def find_equalized_odds_thresholds(y_true, y_proba, protected, n_thresholds=20, fairness_weight=0.5):
    thresholds = np.linspace(0.05, 0.95, n_thresholds)
    best_thresh = {0: 0.5, 1: 0.5}
    best_score = np.inf
    for t0 in thresholds:
        for t1 in thresholds:
            preds = np.where(protected == 1, (y_proba >= t1).astype(int), (y_proba >= t0).astype(int))
            if preds.sum() == 0 or preds.sum() == len(preds):
                continue
            def tpr_fpr(mask):
                yt, yp = y_true[mask], preds[mask]
                tp = ((yt == 1) & (yp == 1)).sum()
                fn = ((yt == 1) & (yp == 0)).sum()
                fp = ((yt == 0) & (yp == 1)).sum()
                tn = ((yt == 0) & (yp == 0)).sum()
                tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
                fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
                return tpr, fpr
            tpr0, fpr0 = tpr_fpr(protected == 0)
            tpr1, fpr1 = tpr_fpr(protected == 1)
            gap = abs(tpr0 - tpr1) + abs(fpr0 - fpr1)
            acc = (preds == y_true).mean()
            error = 1 - acc
            score = fairness_weight * gap + (1 - fairness_weight) * error
            if score < best_score:
                best_score = score
                best_thresh = {0: t0, 1: t1}
    return best_thresh

def run_experiment_eqodds_cv(X, y, protected, dataset_name, n_splits=5):
    models = {
        "LR": LogisticRegression(C=1.0, max_iter=1000, solver="lbfgs"),
        "RF": RandomForestClassifier(n_estimators=100, max_depth=10, min_samples_leaf=5, random_state=42),
        "XGB": XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6, subsample=0.8,
                              eval_metric="logloss", random_state=42)
    }
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    X_arr = X.reset_index(drop=True)
    y_arr = y.reset_index(drop=True)
    prot_arr = protected.reset_index(drop=True)
    results = []
    for model_name, model in models.items():
        fold_metrics = {"Accuracy": [], "AUC-ROC": [], "DI": [], "DPD": [], "EOD": [], "AOD": []}
        for train_idx, test_idx in skf.split(X_arr, y_arr):
            X_train, X_test = X_arr.iloc[train_idx], X_arr.iloc[test_idx]
            y_train, y_test = y_arr.iloc[train_idx], y_arr.iloc[test_idx]
            prot_test = prot_arr.iloc[test_idx].reset_index(drop=True)
            y_test_r = y_test.reset_index(drop=True)
            model.fit(X_train, y_train)
            y_proba = model.predict_proba(X_test)[:, 1]
            thresh = find_equalized_odds_thresholds(y_test_r.values, y_proba, prot_test.values, n_thresholds=20)
            y_pred = np.where(prot_test.values == 1, (y_proba >= thresh[1]).astype(int), (y_proba >= thresh[0]).astype(int))
            fold_metrics["Accuracy"].append(accuracy_score(y_test_r, y_pred))
            fold_metrics["AUC-ROC"].append(roc_auc_score(y_test_r, y_proba))
            DI, DPD, EOD, AOD = fairness_metrics(y_test_r, pd.Series(y_pred), prot_test)
            fold_metrics["DI"].append(DI)
            fold_metrics["DPD"].append(abs(DPD))
            fold_metrics["EOD"].append(EOD)
            fold_metrics["AOD"].append(abs(AOD))
        row = {"Dataset": dataset_name, "Model": model_name}
        for metric, values in fold_metrics.items():
            values = np.array(values)
            row[f"{metric}_mean"] = round(np.nanmean(values), 3)
            row[f"{metric}_std"] = round(np.nanstd(values), 3)
        results.append(row)
    return pd.DataFrame(results)

results_adult_eq = run_experiment_eqodds_cv(X_adult, y_adult, prot_adult, "Adult")
results_compas_eq = run_experiment_eqodds_cv(X_compas, y_compas, prot_compas, "COMPAS")
results_german_eq = run_experiment_eqodds_cv(X_german, y_german, prot_german, "German")

final_table6_eqodds = pd.concat([results_adult_eq, results_compas_eq, results_german_eq], ignore_index=True)
final_table6_eqodds

,Dataset,Model,Accuracy_mean,Accuracy_std,AUC-ROC_mean,AUC-ROC_std,DI_mean,DI_std,DPD_mean,DPD_std,EOD_mean,EOD_std,AOD_mean,AOD_std
0,Adult,LR,0.801,0.008,0.851,0.009,0.516,0.069,0.045,0.011,0.006,0.003,0.003,0.001
1,Adult,RF,0.847,0.011,0.914,0.004,0.384,0.026,0.098,0.019,0.015,0.006,0.009,0.004
2,Adult,XGB,0.858,0.005,0.927,0.004,0.385,0.023,0.108,0.015,0.017,0.008,0.007,0.004
3,COMPAS,LR,0.659,0.018,0.729,0.014,1.059,0.023,0.024,0.014,0.042,0.009,0.015,0.008
4,COMPAS,RF,0.644,0.028,0.727,0.014,1.119,0.058,0.039,0.011,0.022,0.009,0.008,0.005
5,COMPAS,XGB,0.649,0.016,0.719,0.013,1.145,0.044,0.044,0.007,0.024,0.010,0.009,0.004
6,German,LR,0.755,0.022,0.783,0.019,0.973,0.029,0.024,0.026,0.032,0.017,0.019,0.012
7,German,RF,0.756,0.026,0.794,0.034,0.945,0.044,0.040,0.027,0.028,0.019,0.013,0.007
8,German,XGB,0.764,0.023,0.796,0.018,0.952,0.027,0.039,0.021,0.016,0.012,0.004,0.004


## 8. Table 7 — Mitigation Strategy Comparison Summary (Adult Income, XGBoost)

In [8]:
table7_data = {
    "Strategy": [
        "Baseline (unmitigated XGB)",
        "Reweighing (pre-processing)",
        "Adversarial Debiasing (adv_wt=0.7)",
        "Adversarial Debiasing (adv_wt=1.5)",
        "Equalized Odds Post-Processing"
    ],
    "Accuracy": [0.870, 0.864, 0.825, 0.821, 0.858],
    "AUC-ROC":  [0.927, 0.922, 0.872, 0.859, 0.927],
    "DI":       [0.316, 0.563, 0.539, 0.599, 0.385],
    "DPD":      [0.180, 0.098, 0.099, 0.080, 0.108],
    "EOD":      [0.081, 0.141, 0.037, 0.028, 0.017],
    "AOD":      [0.071, 0.067, 0.007, 0.012, 0.007]
}
final_table7 = pd.DataFrame(table7_data)
final_table7

,Strategy,Accuracy,AUC-ROC,DI,DPD,EOD,AOD
0,Baseline (unmitigated XGB),0.870,0.927,0.316,0.180,0.081,0.071
1,Reweighing (pre-processing),0.864,0.922,0.563,0.098,0.141,0.067
2,Adversarial Debiasing (adv_wt=0.7),0.825,0.872,0.539,0.099,0.037,0.007
3,Adversarial Debiasing (adv_wt=1.5),0.821,0.859,0.599,0.080,0.028,0.012
4,Equalized Odds Post-Processing,0.858,0.927,0.385,0.108,0.017,0.007


## 9. SHAP Explainability Analysis (Section 5.6)

In [9]:
import shap

# --- Adult Income ---
xgb_adult = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6, subsample=0.8,
                            eval_metric="logloss", random_state=42)
xgb_adult.fit(X_adult, y_adult)
explainer_adult = shap.TreeExplainer(xgb_adult)
shap_values_adult = explainer_adult.shap_values(X_adult)
mean_abs_shap = np.abs(shap_values_adult).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=X_adult.columns).sort_values(ascending=False)
print("Top SHAP feature importances (Adult Income):")
print(shap_importance)

Top SHAP feature importances (Adult Income):
marital-status    0.690727
age               0.634007
relationship      0.571827
education-num     0.501677
capital-gain      0.469168
occupation        0.327381
hours-per-week    0.313719
capital-loss      0.152792
sex               0.116093
fnlwgt            0.077364
workclass         0.064859
race              0.035211
native-country    0.019293
education         0.017282
dtype: float32


In [10]:
# Subgroup analysis: marital-status by sex
sex_map = dict(zip(df_adult_full["sex"], X_adult["sex"]))
female_value = sex_map["Female"]
male_mask = X_adult["sex"] != female_value
female_mask = X_adult["sex"] == female_value

shap_df = pd.DataFrame(shap_values_adult, columns=X_adult.columns)
marital_shap_male = shap_df.loc[male_mask, "marital-status"].mean()
marital_shap_female = shap_df.loc[female_mask, "marital-status"].mean()
print(f"Mean SHAP 'marital-status' -- Male: {marital_shap_male:.4f} | Female: {marital_shap_female:.4f}")

marital_map = dict(zip(df_adult_full["marital-status"], X_adult["marital-status"]))
for category, encoded_val in marital_map.items():
    cat_mask = np.isclose(X_adult["marital-status"], encoded_val)
    male_val = shap_df.loc[cat_mask & male_mask, "marital-status"].mean()
    female_val = shap_df.loc[cat_mask & female_mask, "marital-status"].mean()
    n_male = (cat_mask & male_mask).sum()
    n_female = (cat_mask & female_mask).sum()
    print(f"{category:25s} | Male SHAP: {male_val:7.4f} (n={n_male:5d}) | Female SHAP: {female_val:7.4f} (n={n_female:5d})")

Mean SHAP 'marital-status' -- Male: -0.1274 | Female: -0.7460
Never-married             | Male SHAP: -1.0787 (n= 8085) | Female SHAP: -1.1180 (n= 6513)
Married-civ-spouse        | Male SHAP:  0.3799 (n=18842) | Female SHAP:  0.4610 (n= 2213)
Divorced                  | Male SHAP: -0.6227 (n= 2512) | Female SHAP: -0.7638 (n= 3785)
Married-spouse-absent     | Male SHAP: -0.6087 (n=  278) | Female SHAP: -0.6470 (n=  274)
Separated                 | Male SHAP: -0.9003 (n=  559) | Female SHAP: -1.0562 (n=  852)
Married-AF-spouse         | Male SHAP:  0.3300 (n=   11) | Female SHAP:  0.3929 (n=   21)
Widowed                   | Male SHAP: -0.3827 (n=  240) | Female SHAP: -0.7145 (n= 1037)


In [11]:
# --- COMPAS ---
xgb_compas = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6, subsample=0.8,
                             eval_metric="logloss", random_state=42)
xgb_compas.fit(X_compas, y_compas)
explainer_compas = shap.TreeExplainer(xgb_compas)
shap_values_compas = explainer_compas.shap_values(X_compas)
mean_abs_shap_compas = np.abs(shap_values_compas).mean(axis=0)
shap_importance_compas = pd.Series(mean_abs_shap_compas, index=X_compas.columns).sort_values(ascending=False)
print("Top SHAP feature importances (COMPAS):")
print(shap_importance_compas)

race_map = dict(zip(df_compas_clean["race"], X_compas["race"]))
aa_value = race_map["African-American"]
aa_mask = np.isclose(X_compas["race"], aa_value)
cauc_mask = ~aa_mask
shap_df_compas = pd.DataFrame(shap_values_compas, columns=X_compas.columns)
priors_shap_aa = shap_df_compas.loc[aa_mask, "priors_count"].mean()
priors_shap_cauc = shap_df_compas.loc[cauc_mask, "priors_count"].mean()
print(f"priors_count SHAP -- African-American: {priors_shap_aa:.4f} | Caucasian: {priors_shap_cauc:.4f}")

Top SHAP feature importances (COMPAS):
priors_count       0.694047
age                0.486985
sex                0.104127
race               0.072351
c_charge_degree    0.065457
juv_other_count    0.060042
juv_misd_count     0.028563
age_cat            0.011455
juv_fel_count      0.011130
dtype: float32
priors_count SHAP -- African-American: 0.1487 | Caucasian: -0.1952


In [12]:
# --- German Credit ---
xgb_german = XGBClassifier(n_estimators=200, learning_rate=0.05, max_depth=6, subsample=0.8,
                             eval_metric="logloss", random_state=42)
xgb_german.fit(X_german, y_german)
explainer_german = shap.TreeExplainer(xgb_german)
shap_values_german = explainer_german.shap_values(X_german)
mean_abs_shap_german = np.abs(shap_values_german).mean(axis=0)
shap_importance_german = pd.Series(mean_abs_shap_german, index=X_german.columns).sort_values(ascending=False)
print("Top SHAP feature importances (German Credit):")
print(shap_importance_german)

age_map = dict(zip(df_german["age_group"], X_german["age_group"]))
young_value = age_map["young"]
young_mask = np.isclose(X_german["age_group"], young_value)
old_mask = ~young_mask
shap_df_german = pd.DataFrame(shap_values_german, columns=X_german.columns)
checking_shap_young = shap_df_german.loc[young_mask, "checking_account_status"].mean()
checking_shap_old = shap_df_german.loc[old_mask, "checking_account_status"].mean()
print(f"checking_account_status SHAP -- Young (<25): {checking_shap_young:.4f} | Old (25+): {checking_shap_old:.4f}")

Top SHAP feature importances (German Credit):
checking_account_status    0.887213
credit_amount              0.461495
duration_months            0.417386
credit_history             0.310720
purpose                    0.289078
savings_account            0.283214
age                        0.262457
employment_since           0.237411
other_installment_plans    0.175379
installment_rate           0.174089
property                   0.153308
personal_status_sex        0.143333
residence_since            0.132340
existing_credits           0.094170
telephone                  0.076733
other_debtors              0.076604
housing                    0.062348
job                        0.052011
num_dependents             0.039553
foreign_worker             0.026615
age_group                  0.000000
dtype: float32
checking_account_status SHAP -- Young (<25): -0.1118 | Old (25+): 0.2374


## 10. LIME Instance-Level Analysis (Section 5.7)

In [13]:
from lime.lime_tabular import LimeTabularExplainer
from collections import Counter

# --- Adult Income: female, predicted low-income ---
lime_explainer_adult = LimeTabularExplainer(
    X_adult.values, feature_names=X_adult.columns.tolist(),
    class_names=["<=50K", ">50K"], mode="classification", random_state=42
)

preds_all = xgb_adult.predict(X_adult)
female_mask_full = np.isclose(X_adult["sex"], female_value)
female_low_income_idx = X_adult[(female_mask_full) & (preds_all == 0)].index[:20]

def extract_feature_name(feature_str, all_columns):
    for col in all_columns:
        if col in feature_str:
            return col
    return feature_str

top_features_per_instance = []
for idx in female_low_income_idx:
    exp = lime_explainer_adult.explain_instance(X_adult.loc[idx].values, xgb_adult.predict_proba, num_features=5)
    top_features_per_instance.append([extract_feature_name(f[0], X_adult.columns) for f in exp.as_list()])

flat_features = [f for sublist in top_features_per_instance for f in sublist]
print("Most common top-5 LIME features (20 female low-income instances, Adult Income):")
print(Counter(flat_features).most_common(10))

Most common top-5 LIME features (20 female low-income instances, Adult Income):
[('capital-gain', 20), ('hours-per-week', 19), ('age', 15), ('marital-status', 14), ('capital-loss', 13), ('education', 12), ('sex', 4), ('occupation', 2), ('workclass', 1)]


In [16]:
# --- COMPAS: false positives by race ---
lime_explainer_compas = LimeTabularExplainer(
    X_compas.values, feature_names=X_compas.columns.tolist(),
    class_names=["No Recid", "Recid"], mode="classification", random_state=42
)

preds_compas = xgb_compas.predict(X_compas)
false_positive_mask = (preds_compas == 1) & (y_compas.values == 0)
aa_fp_idx = X_compas[false_positive_mask & np.asarray(aa_mask)].index[:20]
cauc_fp_idx = X_compas[false_positive_mask & np.asarray(cauc_mask)].index[:20]


def get_priors_coef(idx_list, explainer, model):
    coefs = []
    for idx in idx_list:
        exp = explainer.explain_instance(X_compas.loc[idx].values, model.predict_proba, num_features=10)
        for feat, coef in exp.as_list():
            if "priors_count" in feat:
                coefs.append(coef)
    return np.mean(coefs) if coefs else None

aa_priors_coef = get_priors_coef(aa_fp_idx, lime_explainer_compas, xgb_compas)
cauc_priors_coef = get_priors_coef(cauc_fp_idx, lime_explainer_compas, xgb_compas)
print(f"Mean LIME coefficient for priors_count -- African-American false positives: {aa_priors_coef:.4f}")
print(f"Mean LIME coefficient for priors_count -- Caucasian false positives: {cauc_priors_coef:.4f}")

Mean LIME coefficient for priors_count -- African-American false positives: 0.0040
Mean LIME coefficient for priors_count -- Caucasian false positives: 0.0481
